# Phase 3 — Train & evaluate all methods (Colab T4)

Trains every method on the **suppression-relevant `default` regime** (both regions) and emits the results tables for the paper. Runs the proposed **HierComm** model plus baselines (MAPPO, QMIX, CommNet) and heuristics (No-Op, Value-First, Greedy-Risk, Local-Reactive).

**Runtime → T4 GPU, High-RAM.** The bottleneck is the Cell2Fire subprocess (CPU, single-thread), so the T4 mainly helps the networks; budget **~1–2 h per learned method**. The run is **idempotent and Drive-backed**: if the session drops, just re-run the cells — finished checkpoints are reused and training resumes.

**Before running:** set `REPO_URL`/`BRANCH` below to your pushed branch (Phases 1–2 must be committed & pushed).

In [ ]:
# 0. Sanity: GPU + RAM
!nvidia-smi -L
import psutil; print(f"RAM: {psutil.virtual_memory().total/1e9:.1f} GB")

In [ ]:
# 1. Config — EDIT THESE
REPO_URL = "https://github.com/aliakarma/wildfire-rl.git"
BRANCH   = "additional"     # branch with Phases 1-2 pushed
TRAIN_STEPS = 100000         # per learned method (lower to ~60000 if short on time)
EPISODES    = 15             # eval episodes per seed group
REGIONS     = "saudi,california"
METHODS     = "noop,value_first,greedy_risk,local_reactive,mappo,qmix,commnet,hiercomm,hiercomm_heur"

In [ ]:
# 2. Mount Drive (persists checkpoints/results across sessions -> resumable)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/wildfire_phase3'
import os; os.makedirs(OUT, exist_ok=True); print('Results ->', OUT)

In [ ]:
# 3. Clone (or update) the repo
import os
if not os.path.exists('/content/wildfire-rl'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/wildfire-rl
else:
    !cd /content/wildfire-rl && git fetch --depth 1 origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/wildfire-rl
!git log --oneline -1

In [ ]:
# 4. Build the Cell2Fire binary (skips if it already runs)
import subprocess
BIN = 'third_party/firehose/cell2fire/Cell2FireC/Cell2Fire'
def runs():
    try:
        return subprocess.run([f'./{BIN}', '--help'], capture_output=True, timeout=10).returncode is not None and os.path.exists(BIN)
    except Exception:
        return False
if not runs():
    !sudo apt-get -qq update && sudo apt-get -qq install -y libboost-all-dev libeigen3-dev >/dev/null
    !cd third_party/firehose/cell2fire/Cell2FireC && make -f Makefile_UBUNTU EIGENDIR=/usr/include/eigen3/ 2>&1 | tail -3
!ls -la {BIN} && file {BIN}

In [ ]:
# 5. Install the package (keep Colab's CUDA torch; add only what's missing)
!pip install -q gymnasium==1.0.0 pettingzoo==1.26.1
!pip install -q -e . --no-deps
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
import wildfire_marl; print('wildfire_marl OK')

In [ ]:
# 6. Smoke test (few minutes) — validates the full train->eval->table path end-to-end
!python scripts/run_phase3.py --quick --regions saudi --methods mappo,hiercomm,value_first,noop --out {OUT}_smoke
print(open(f'{OUT}_smoke/phase3_main_table.tex').read())

In [ ]:
# 7. FULL run — idempotent & resumable (re-run this cell if the session drops)
!python scripts/run_phase3.py --regions {REGIONS} --methods {METHODS} \
    --train-steps {TRAIN_STEPS} --episodes {EPISODES} --out {OUT}

## Deterministic regeneration (reproducibility)
The frozen table's flat baselines (MAPPO, CommNet) were trained with an **unseeded** environment, so their fire scenarios came from OS entropy and were not reproducible. The seeding fix is in the pulled branch. The cell below regenerates **MAPPO, CommNet, HierComm, HierComm-heur** deterministically on **CPU** (proven bit-identical), keeps the invariant **QMIX**, and compares OLD vs NEW. Idempotent + Drive-backed, so it resumes if the session drops.

In [ ]:
# 7b. DETERMINISTIC REGENERATION (reproducibility fix)
# Non-invariant methods only; QMIX (invariant, No-Op) is copied from the frozen run.
# CPU is forced: determinism is proven bit-identical on CPU; GPU cuDNN can reintroduce
# nondeterminism, and Cell2Fire is CPU-bound so GPU gives ~no speedup here.
import os, shutil
OUT_REPRO = f'{OUT}_repro'
os.makedirs(OUT_REPRO, exist_ok=True)
for r in ['saudi', 'california']:
    src = f'{OUT}/checkpoint_qmix_{r}.pt'
    if os.path.exists(src):
        shutil.copy(src, OUT_REPRO)
!CUDA_VISIBLE_DEVICES='' python scripts/run_phase3.py \
    --methods mappo,qmix,commnet,hiercomm,hiercomm_heur \
    --regions {REGIONS} --train-steps {TRAIN_STEPS} --episodes {EPISODES} --out {OUT_REPRO}
print(open(f'{OUT_REPRO}/phase3_main_table.tex').read())

In [ ]:
# 7c. OLD (frozen) vs NEW (deterministic) comparison + significance
import json
old = json.load(open(f'{OUT}/phase3_summary.json'))
new = json.load(open(f'{OUT_REPRO}/phase3_summary.json'))
for region in new['regions']:
    print(f'\n=== {region.upper()}  (WEL: old -> new) ===')
    rows = new['regions'][region]
    for k, v in rows.items():
        if k == 'comparisons':
            continue
        o = old['regions'].get(region, {}).get(k, {}).get('metrics', {}).get('WEL', {}).get('mean')
        n = v.get('metrics', {}).get('WEL', {}).get('mean')
        if o is not None and n is not None:
            print(f"  {v['label']:<22} {o:6.2f} -> {n:6.2f}")
    print('  HierComm vs baselines (NEW):')
    for k, c in rows.get('comparisons', {}).get('WEL', {}).items():
        if k.startswith('hiercomm_vs'):
            print(f"     {k}: p={c['p']:.4f} d={c['d']:.2f}")

In [ ]:
# 8. Inspect results
import json
print(open(f'{OUT}/phase3_main_table.tex').read())
s = json.load(open(f'{OUT}/phase3_summary.json'))
for region, reg in s['regions'].items():
    print(f"\n=== {region.upper()} : HierComm vs baselines (WEL) ===")
    for k, v in reg.get('comparisons', {}).get('WEL', {}).items():
        if k.startswith('hiercomm_vs'):
            print(f"  {k}: t={v['t']:.2f} p={v['p']:.4f} d={v['d']:.2f}")

In [ ]:
# 9. (Optional) also download a zip of results (already persisted to Drive)
!cd {OUT} && zip -qr /content/phase3_results.zip . -x '*.pt'
from google.colab import files; files.download('/content/phase3_results.zip')